In [22]:
import pandas as pd 
import numpy as np 
import matplotlib 
from pathlib import Path

# Model Selection and Evaluation
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# for hyperparam tuning
from sklearn.model_selection import GridSearchCV

In [23]:
data_path = Path('../data/preprocessed_pokemon_data.csv')
pkmn_df = pd.read_csv(data_path)
seed = 42

In [24]:
target = ['tier']
X = pkmn_df.drop(columns=[t for t in target])
y = pkmn_df[target]

# 80% train 20% test, stratify is to make sure the tier distribution is more even in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

In [27]:
'''
NOTE: objective: multi-softclass essentially tells the model it is doing a classification problem
given multiple classes like OU, UU, etc. choose the most fitting one and output; softmax converts tier scores->probabilities

'''

tier_model = XGBClassifier(
    n_estimators = 200,
    learning_rate = 0.1,
    max_depth=6,
    colsample_bytree=0.3,
    num_class=len(np.unique(y))
)

tier_model.fit(X_train, y_train)
predictions = tier_model.predict(X_test)


print(classification_report(y_test, predictions, zero_division=0))

mae = mean_absolute_error(y_test, predictions)
print(mae)


              precision    recall  f1-score   support

           0       0.85      0.73      0.79        45
           1       0.93      1.00      0.96        64
           2       0.83      0.63      0.72        30
           3       0.70      0.91      0.79        93
           4       0.00      0.00      0.00         5
           5       0.00      0.00      0.00         7
           6       0.00      0.00      0.00         5
           7       0.00      0.00      0.00         8
           8       0.60      0.50      0.55        12
           9       0.00      0.00      0.00         1

    accuracy                           0.77       270
   macro avg       0.39      0.38      0.38       270
weighted avg       0.72      0.77      0.74       270

0.6037037037037037


In [26]:
tier_rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=15, 
    max_features='sqrt',
    random_state=seed,
    class_weight='balanced' # chose because some tiers have very few mons
)

tier_rf.fit(X_train, y_train.values.ravel())
rf_predictions = tier_rf.predict(X_test)

# 4. EVALUATE
print("--- Random Forest Tier Prediction Results ---")
print(classification_report(y_test, rf_predictions, zero_division=0))

print(mean_absolute_error(y_test, rf_predictions))

--- Random Forest Tier Prediction Results ---
              precision    recall  f1-score   support

           0       0.76      0.71      0.74        45
           1       0.89      1.00      0.94        64
           2       0.81      0.57      0.67        30
           3       0.65      0.87      0.74        93
           4       0.00      0.00      0.00         5
           5       0.00      0.00      0.00         7
           6       0.00      0.00      0.00         5
           7       0.00      0.00      0.00         8
           8       0.75      0.50      0.60        12
           9       0.00      0.00      0.00         1

    accuracy                           0.74       270
   macro avg       0.39      0.36      0.37       270
weighted avg       0.68      0.74      0.70       270

0.7148148148148148


In [32]:
print(list(zip(tier_order, [i for i in range(10)])))

[('Illegal', 0), ('LC', 1), ('NFE', 2), ('RU', 3), ('RUBL', 4), ('UU', 5), ('UUBL', 6), ('OU', 7), ('Uber', 8), ('AG', 9)]
